# Breakout Room #2 — La matriz cliente × producto y la similitud

**Objetivo:**
- Armar la matriz de compras (una fila por cliente, una columna por producto).
- Entender qué significa que dos clientes, o dos productos, "se parezcan" (similitud coseno).
- Ver con datos por qué la matriz es el corazón del filtrado colaborativo, y dónde falla.

Es la **base** del siguiente bloque: ahí usaremos esta matriz para recomendar. Donde dice **`# TODO`** hay algo para que ustedes completen.

> **Similitud coseno** entre dos vectores a y b: `a · b / (|a| × |b|)`. Va de 0 (no comparten nada) a 1 (idénticos).
> Con compras de sí/no, es *qué tanto se solapan* dos listas de compras, ajustado por su tamaño.

In [ ]:
# En Google Colab no hace falta instalar nada extra.
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
VERDE, OSCURO, NARANJA, GRIS = "#00704A", "#11241C", "#C4512F", "#9A988E"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})

orders = pd.read_csv("data/orders.csv", parse_dates=["order_date"])
items = pd.read_csv("data/items.csv")
customers = pd.read_csv("data/customers.csv")

# Escondemos los últimos 90 días: son el "examen" del último breakout. Hoy trabajamos solo con el pasado.
CUTOFF = orders["order_date"].max() - pd.Timedelta(days=90)
train = orders[orders["order_date"] <= CUTOFF]
print(f"Corte: {CUTOFF.date()} · {len(train):,} líneas de compra antes del corte")

## Paso 1 — la matriz

Una fila por cliente, una columna por producto. `1` = el cliente compró ese producto **al menos una vez** antes del corte.
Incluimos a los 2,000 clientes, aunque algunos no hayan comprado nada antes del corte.

In [ ]:
M = (pd.crosstab(train["customer_id"], train["item"]) > 0).astype(int)
M = M.reindex(customers["customer_id"], fill_value=0)          # los 2,000 clientes
print("Forma de la matriz:", M.shape)
print(f"Densidad: {M.values.mean():.1%} de las celdas tienen un 1  →  {1 - M.values.mean():.1%} están vacías")
n_prod = M.sum(axis=1)
print(f"Productos distintos por cliente: promedio {n_prod.mean():.1f} · "
      f"con 0 productos: {(n_prod == 0).sum()} · con 1 solo: {(n_prod == 1).sum()}")
M.iloc[:6, :8]

## Paso 2 — similitud entre dos clientes (a mano)

Tomamos tres clientes: **A = 14**, **B = 1015** y **C = 62**. Primero miren qué compró cada uno, y adivinen
quién se parece a quién *antes* de calcular nada.

In [ ]:
ID_A, ID_B, ID_C = 14, 1015, 62
for cid in (ID_A, ID_B, ID_C):
    print(f"Cliente {cid:>4}:", ", ".join(M.columns[M.loc[cid] == 1]))

In [ ]:
a, b_, c = M.loc[ID_A].values, M.loc[ID_B].values, M.loc[ID_C].values

# TODO 1: completen la función coseno(x, y) = (x · y) / (|x| × |y|).
# Pistas: np.dot(x, y) para el producto punto; np.linalg.norm(x) para la longitud del vector.
def coseno(x, y):
    return ...        # <-- reemplacen esto

print(f"sim(A, B) = {coseno(a, b_):.3f}")
print(f"sim(A, C) = {coseno(a, c):.3f}")
print("Revisión con scikit-learn:", cosine_similarity([a], [b_])[0, 0].round(3), cosine_similarity([a], [c])[0, 0].round(3))

## Paso 3 — la misma idea, pero entre productos

Los productos también tienen "vector": la **columna** de la matriz (qué clientes lo compraron). Calculen la similitud entre
**Latte y Croissant**, y entre **Latte y Ensalada**.

In [ ]:
latte, croissant, ensalada = M["Latte"].values, M["Croissant"].values, M["Ensalada"].values

# TODO 2: calculen la similitud coseno entre las columnas, usando SU función coseno() del Paso 2.
sim_latte_croissant = ...     # <-- reemplacen esto
sim_latte_ensalada  = ...     # <-- reemplacen esto
print(f"sim(Latte, Croissant) = {sim_latte_croissant:.3f}")
print(f"sim(Latte, Ensalada)  = {sim_latte_ensalada:.3f}")

## Paso 4 — todas las similitudes, de una vez

Con `cosine_similarity` calculamos todas las parejas de productos (29 × 29). Ordenamos los productos por "familia de gusto"
para que se vea si hay estructura.

In [ ]:
orden = ["Latte", "Cappuccino", "Croissant", "Muffin", "Scone", "Dona",
         "Cold Brew", "Matcha Latte", "Ensalada", "Granola", "Copa de Fruta", "Yogur",
         "Americano", "Espresso", "Flat White", "Sándwich", "Panini", "Wrap", "Bagel",
         "Frappuccino", "Refresher", "Galleta", "Pastel de Queso", "Brownie",
         "Té Chai", "Té Verde", "Banana Bread", "Chocolate Caliente", "Agua"]
S_item = pd.DataFrame(cosine_similarity(M[orden].values.T), index=orden, columns=orden)

fig, ax = plt.subplots(figsize=(8.5, 7.5))
im = ax.imshow(S_item.values, cmap="Greens", vmin=0, vmax=0.6)
ax.set_xticks(range(len(orden))); ax.set_xticklabels(orden, rotation=90, fontsize=8)
ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden, fontsize=8)
ax.set_title("Similitud entre productos (más verde = se compran por los mismos clientes)")
plt.colorbar(im, shrink=0.7); plt.tight_layout(); plt.show()

**Para discutir:**
1. ¿Qué bloques ven en la gráfica? ¿Qué tipo de clientes creen que representa cada bloque?
2. Miren la fila y la columna del **Agua** (la última). ¿Qué pasa y por qué? (Pista: recuerden el Paso 5 del breakout anterior.)

## Paso 5 — el cliente sin historia

La matriz solo sabe lo que los clientes ya compraron. Veamos a quién no le sirve.

In [ ]:
dist = n_prod.clip(upper=5).value_counts().sort_index()
print((dist / len(M)).rename("% de clientes").round(3).to_string())
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(dist.index.astype(str).str.replace("5", "5+"), dist.values / len(M) * 100, color=VERDE)
ax.set_xlabel("Productos distintos comprados antes del corte"); ax.set_ylabel("% de clientes")
plt.tight_layout(); plt.show()

3. ¿A cuántos clientes les sirve poco o nada la similitud (0 o 1 producto)? ¿Qué le recomendarían a un cliente con 0 productos?
4. Ahora que ven la matriz, ¿qué le pasa a la similitud si el catálogo tuviera 10,000 productos y 2 millones de clientes? (Lo retomamos en el siguiente breakout.)